# Preparation

In [ ]:
import logging
import time
from itertools import combinations
from pathlib import Path

import myo
import myoktros
import numpy as np
import seaborn as sn
from sklearn.metrics import accuracy_score, confusion_matrix

EXTRA_GESTURES = [
    "STRETCH_FINGERS",
    "EXTENSION",
    "HORN",
    "FLEMING",
    "FLEXION",
    "TENNET",
    "SHOOT",
]
# gesture combinations
gesture_combinations = []
for n in range(1, len(EXTRA_GESTURES)):
    for c in combinations(EXTRA_GESTURES, n):
        gesture_combinations.append(('REST', 'GRAB') + c)

# global variables
arm_dominance = "right"
assets_path = Path.cwd() / "assets"
config_path = Path.cwd() / "config.ini"
data_path = Path.cwd() / "data"
emg_mode = myo.types.EMGMode.SEND_FILT
n_samples = 25
knn_k = 3
knn_algorithm = "auto"
knn_metric = "minkowski"
svm_c = 1.0
svm_degree = 3
svm_gamma = "scale"
svm_kernel = "rbf"
test_data_path = Path.cwd() / "tests" / "data"

# configurations
myoktros.Gesture.load_config(config_path)

# options
np.set_printoptions(precision=3, suppress=True)

# Train the models

## keras

In [ ]:
ksm = myoktros.KerasSequentialModel.fit(
    arm_dominance,
    assets_path,
    data_path,
    emg_mode,
    n_samples,
)

In [ ]:
# load the model from file
ksm = myoktros.KerasSequentialModel(
    arm_dominance,
    assets_path,
    emg_mode,
    n_samples,
).model

## knn

In [ ]:
knn = myoktros.KNNClassifier.fit(
    arm_dominance,
    assets_path,
    data_path,
    emg_mode,
    knn_k,
    knn_algorithm,
    knn_metric,
    n_samples,
)

In [ ]:
# load the model from file
knn = myoktros.KNNClassifier(
    arm_dominance,
    assets_path,
    emg_mode,
    knn_k,
    knn_metric,
    n_samples,
).model

## svm

In [ ]:
svm = myoktros.SVMClassifier.fit(
    arm_dominance,
    assets_path,
    data_path,
    emg_mode,
    n_samples,
    svm_c,
    svm_degree,
    svm_gamma,
    svm_kernel,
)

In [ ]:
# load the model from file
svm = myoktros.SVMClassifier(
    arm_dominance,
    assets_path,
    emg_mode,
    n_samples,
    svm_c,
    svm_degree,
    svm_gamma,
    svm_kernel,
).model

# Visualize the classification performance vs. test data

In [ ]:
# keras
x_test = myoktros.GestureModel.read_data_agg(test_data_path, arm_dominance, emg_mode, n_samples)
y_test = x_test.pop('gesture')

predictions = ksm.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# plot heatmap
ax = sn.heatmap(cm, annot=True, cmap="Blues", xticklabels=myoktros.Gesture.names, yticklabels=myoktros.Gesture.names)
_ = ax.set(xlabel="Predicted", ylabel="Actual")
# _ = ax.xaxis.tick_top()

In [ ]:
# knn
x_test = myoktros.GestureModel.read_data_agg(test_data_path, arm_dominance, emg_mode, n_samples)
y_test = x_test.pop('gesture')

predicted_labels = knn.predict(x_test)
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# plot heatmap
ax = sn.heatmap(cm, annot=True, cmap="Blues", xticklabels=myoktros.Gesture.names, yticklabels=myoktros.Gesture.names)
_ = ax.set(xlabel="Predicted", ylabel="Actual")
# _ = ax.xaxis.tick_top()

In [ ]:
# svm
x_test = myoktros.GestureModel.read_data_agg(test_data_path, arm_dominance, emg_mode, n_samples)
y_test = x_test.pop('gesture')

predicted_labels = svm.predict(x_test)
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# plot heatmap
ax = sn.heatmap(cm, annot=True, cmap="Blues", xticklabels=myoktros.Gesture.names, yticklabels=myoktros.Gesture.names)
_ = ax.set(xlabel="Predicted", ylabel="Actual")
# _ = ax.xaxis.tick_top()

# Find out the optimal configurations for k-NN model

In [ ]:
results = []
for i, c in enumerate(gesture_combinations):
    myoktros.Gesture.load_list(c)
    trials = []
    for knn_metric in ["minkowski", "euclidean", "manhattan"]:
        for k in range(1, 16, 2):
            for n_samples in range(25, 201, 25):
                # fit the model
                knn = myoktros.KNNClassifier.fit(
                    arm_dominance,
                    assets_path,
                    data_path,
                    emg_mode,
                    k,
                    knn_algorithm,
                    knn_metric,
                    n_samples,    
                )
                # get the appropriate test data
                x_test = myoktros.GestureModel.read_data_agg(test_data_path, arm_dominance, emg_mode, n_samples)
                y_test = x_test.pop('gesture')
                # evaluate the accuracy
                predicted_labels = knn.predict(x_test)
                acc = accuracy_score(y_test, predicted_labels)
                trials.append((k, knn_metric, n_samples, acc))
                results.append((c, k, knn_metric, n_samples, acc))

    best = max(trials, key=lambda r: r[3])
    print(f"[{i+1}/{len(gesture_combinations)}] {c} – k: {best[0]}, metric: {best[1]}, n_samples: {best[2]} at {best[3]:.3f}")

# sort the best performing combinations by the accuracy
results.sort(key=lambda r: r[4], reverse=True)

print("10 best performers:")
for i, r in enumerate(results[:10]):
    print(f"{i+1} {r[0]} – k: {r[1]}, metric: {r[2]}, n_samples: {r[3]}, at {r[4]}")

# write out the result to a csv file
now = time.strftime("%Y%m%d%H%M%S")
with (assets_path / f"knn-eval-{now}.csv").open('w') as f:
    print("gestures;k;metric;n_samples;acc", file=f)
    for r in results:
        print(";".join(map(str, (r[0], r[1], r[2], r[3], r[4]))), file=f)

# Find out the optimal configurations for SVM model

In [ ]:
results = []
for i, c in enumerate(gesture_combinations):
    myoktros.Gesture.load_list(c)
    trials = []
    for svm_kernel in ["linear", "poly", "rbf", "sigmoid"]:
        for n_samples in range(25, 201, 25):
            # fit the model
            svm = myoktros.SVMClassifier.fit(
                arm_dominance,
                assets_path,
                data_path,
                emg_mode,
                n_samples,
                svm_c,
                svm_degree,
                svm_gamma,
                svm_kernel,
            )
            # get the appropriate test data
            x_test = myoktros.GestureModel.read_data_agg(test_data_path, arm_dominance, emg_mode, n_samples)
            y_test = x_test.pop('gesture')
            # evaluate the accuracy
            predicted_labels = svm.predict(x_test)
            acc = accuracy_score(y_test, predicted_labels)
            trials.append((acc, n_samples, svm_kernel))
            results.append((c, svm_c, svm_degree, svm_gamma, svm_kernel, n_samples, acc))
    
    best = max(trials, key=lambda r: r[0])
    print(f"[{i+1}/{len(gesture_combinations)}] {c} – acc: {best[0]:.3f} n_samples: {best[1]} kernel: {best[2]}")

# sort the best performing combinations by the accuracy
results.sort(key=lambda r: r[6], reverse=True)

# write out the result to a csv file
now = time.strftime("%Y%m%d%H%M%S")
with (assets_path / f"svm-eval-{now}.csv").open('w') as f:
    print("gestures;c;degree;gamma;kernel;n_samples;acc", file=f)
    for r in results:
        print(";".join(map(str, (r[0], r[1], r[2], r[3], r[4], r[5], r[6]))), file=f)